# PromptGFM-Bio — Kaggle Training Notebook
**Gene-Phenotype Prediction for Rare Diseases**

⚙️ Before running:
- Go to **Settings → Accelerator → GPU T4 x2** (or GPU P100)
- Turn on **Internet** in Settings
- *(Optional)* Add your project GitHub repo URL in the cell below

**Session time limit**: 9 h — checkpoints are saved after every epoch, so you can resume safely.

## 1. Environment Check

In [ ]:
import sys, subprocess, torch, os

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')
    print(f'GPU count: {torch.cuda.device_count()}')
else:
    print('⚠️  No GPU detected! Enable GPU in Notebook Settings.') 

## 2. Install PyTorch Geometric

In [ ]:
# Detect exact torch + CUDA version to pick the right wheel URL
import torch

TORCH_VER = torch.__version__.split('+')[0]        # e.g. '2.1.0'
CUDA_VER  = torch.version.cuda.replace('.', '')    # e.g. '121'
WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'
print(f'Installing PyG wheels from: {WHEEL_URL}')

packages = [
    'torch-scatter',
    'torch-sparse',
    'torch-cluster',
    'torch-spline-conv',
    'torch-geometric',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '-f', WHEEL_URL] + packages,
    check=True
)
print('✅ PyTorch Geometric installed')

In [ ]:
# Install remaining dependencies
extra = [
    'transformers>=4.40.0',
    'sentence-transformers>=2.7.0',
    'biopython>=1.84',
    'networkx>=3.2',
    'wandb>=0.17.0',
    'python-dotenv>=1.0.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + extra, check=True)
print('✅ Extra packages installed')

## 3. Clone Project Code from GitHub

In [ ]:
# ── Edit this URL to point to your GitHub repo ──────────────────────────────
GITHUB_URL = 'https://github.com/pes1ug23am910/PROMPTGMF-Bio-Kaggle.git'
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = '/kaggle/working/PromptGFM-Bio'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, PROJECT_DIR], check=True)
    print(f'✅ Cloned to {PROJECT_DIR}')
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)
    print(f'✅ Pulled latest changes in {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

## 4. Create Directory Structure

In [ ]:
from pathlib import Path

dirs = [
    'data/raw', 'data/processed', 'data/splits',
    'checkpoints/promptgfm_film',
    'logs',
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✅ Directories created')

## 5. Download Biomedical Datasets
Total download: ~1.5 GB (BioGRID + STRING + DisGeNET + HPO).  
This step takes a few minutes. Skip it if you have already preprocessed data stored as a Kaggle Dataset.

In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/download_data.py', '--dataset', 'all'],
    capture_output=False   # show live output
)
print('Download exit code:', result.returncode)

## 6. Preprocess Data (Build Knowledge Graph)

In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/preprocess_all.py'],
    capture_output=False
)
print('Preprocessing exit code:', result.returncode)

# Confirm the graph was built
graph_path = Path('data/processed/biomedical_graph.pt')
if graph_path.exists():
    size_mb = graph_path.stat().st_size / 1e6
    print(f'✅ Graph ready: {graph_path}  ({size_mb:.1f} MB)')
else:
    print('⚠️  Graph file not found — check preprocessing logs above')

## 7. (Optional) Restore Previous Checkpoint
If you have a saved checkpoint from a previous Kaggle session stored as a Kaggle Dataset, copy it here.

In [ ]:
import shutil

# Path to your checkpoint Kaggle Dataset (e.g. /kaggle/input/promptgfm-checkpoints/)
CHECKPOINT_DATASET = '/kaggle/input/promptgfm-checkpoints'   # change if needed

if os.path.exists(CHECKPOINT_DATASET):
    dest = Path('checkpoints/promptgfm_film')
    for f in Path(CHECKPOINT_DATASET).glob('*'):
        shutil.copy2(f, dest / f.name)
    print(f'✅ Checkpoints restored from {CHECKPOINT_DATASET}')
else:
    print('No checkpoint dataset found — starting fresh (this is fine for first run)')

## 8. W&B Login (Optional)

In [ ]:
# Paste your W&B API key here, or set use_wandb: false in the config
WANDB_API_KEY = ''   # leave empty to skip

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print('✅ W&B logged in')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled — training metrics will be logged to stdout only')

## 9. Train
Uses `configs/kaggle_config.yaml` which is tuned for the Kaggle T4 (16 GB VRAM).  
To **resume** from the last saved checkpoint, set `RESUME = True`.

In [ ]:
RESUME = False   # set True to resume from last checkpoint

cmd = [sys.executable, 'scripts/train.py', '--config', 'configs/kaggle_config.yaml']
if RESUME:
    # The resume script auto-selects the latest checkpoint
    cmd = [sys.executable, 'scripts/resume_training.py', '--config', 'configs/kaggle_config.yaml']

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
print('Training exit code:', result.returncode)

## 10. Persist Checkpoints as Kaggle Output
Kaggle saves everything in `/kaggle/working/` as output after the session.  
Add those outputs as a new **Kaggle Dataset** so you can reload them next session.

In [ ]:
import shutil

OUTPUT_DIR = Path('/kaggle/working/saved_checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)

src = Path('checkpoints/promptgfm_film')
for f in src.glob('*'):
    shutil.copy2(f, OUTPUT_DIR / f.name)

print(f'✅ Checkpoints copied to {OUTPUT_DIR}')
print('Files saved:')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

## 11. Quick Evaluation

In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/evaluate.py',
     '--config', 'configs/kaggle_config.yaml',
     '--checkpoint', 'checkpoints/promptgfm_film/best_model.pt'],
    capture_output=False
)
print('Evaluation exit code:', result.returncode)